# Ensemble Forecasts on Two T4 GPUs

Measures how many SFNO ensemble members per minute one T4 and two T4s
can run, with 1, 2 or 4 members per batch
([docs/ensemble-on-two-gpus.md](../docs/ensemble-on-two-gpus.md)). The
changes and their success criteria were written before this notebook was
run.

Run on Kaggle with **Accelerator: GPU T4 x2** and **Internet: on**. The
notebook uses both GPUs and takes about 45 minutes.

The notebook gets the project from GitHub. To use a local copy instead,
upload the repository as a Kaggle dataset named `earth-intelligence`
and add it to the notebook.

Output: `/kaggle/working/ensemble_on_t4.zip`. Unzip it into
`ml/artifacts/ensemble_on_t4/`.

In [ ]:
!pip install -q "makani @ git+https://github.com/NVIDIA/makani.git@b38fcb2799d7dbc146fa60459f3f9823394a8bf1" "earth2studio[data,sfno]==0.20.0" python-dotenv ninja nvidia-ml-py 2>&1 | tail -3

In [ ]:
import shutil
import subprocess
from pathlib import Path

REPOSITORY = "https://github.com/khoahonguyen0210/Earth_intelligence.git"
INPUT = Path("/kaggle/input")
PROJECT = Path("/kaggle/working/earth-intelligence")
OUTPUT = Path("/kaggle/working/ensemble_on_t4")
INITIAL_STATE = Path("/kaggle/working/initial_state.pt")


def uploaded_project():
    for marker in INPUT.rglob("ml/pyproject.toml"):
        return marker.parent.parent
    for archive in INPUT.rglob("earth-intelligence.tar.gz"):
        shutil.unpack_archive(archive, PROJECT)
        return PROJECT
    return None


if not PROJECT.exists():
    source = uploaded_project()
    if source is None:
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORY, str(PROJECT)], check=True
        )
    elif source != PROJECT:
        shutil.copytree(source, PROJECT)

## Download the model once

The first load downloads the SFNO package. Loading it here keeps the
download out of the timed runs.

In [ ]:
!python -c "from earth2studio.models.px import SFNO; SFNO.load_model(SFNO.load_default_package())"

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
!nvidia-smi --query-gpu=index,name,memory.total,clocks.max.sm,power.limit --format=csv | tee {OUTPUT}/gpus.csv
!nproc

## Runs

Each run starts new worker processes, which load the model again. The
first run also downloads the initial state, which the later runs reuse.

In [ ]:
import json

RUNS = [
    ("gpus1_batch1_1", 1, 1),
    ("gpus1_batch2_1", 1, 2),
    ("gpus2_batch1_1", 2, 1),
    ("gpus1_batch1_2", 1, 1),
    ("gpus1_batch2_2", 1, 2),
    ("gpus2_batch1_2", 2, 1),
    ("gpus1_batch4_1", 1, 4),
]
results = {}
for name, gpus, batch in RUNS:
    command = (
        f"cd {PROJECT} && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True PYTHONPATH=ml/src "
        f"python scripts/research/run_ensemble.py --members 8 --gpus {gpus} "
        f"--batch-size {batch} --initial-state {INITIAL_STATE} --output {OUTPUT / name}"
    )
    finished = subprocess.run(command, shell=True, capture_output=True, text=True)
    results[name] = {
        "return_code": finished.returncode,
        "last_lines": (finished.stdout + finished.stderr).splitlines()[-25:],
    }
    print(name, finished.returncode)

(OUTPUT / "runs.json").write_text(json.dumps(results, indent=2))

In [ ]:
!cd {PROJECT} && PYTHONPATH=ml/src python scripts/research/compare_ensemble_runs.py {OUTPUT}

In [ ]:
INITIAL_STATE.unlink(missing_ok=True)
shutil.make_archive("/kaggle/working/ensemble_on_t4", "zip", OUTPUT)